In [25]:
import numpy as np
import pandas as pd
import random
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool


#   Data: one graph (nodes, edges, label)
#   DataLoader: batches multiple graphs together for training
from torch_geometric.data import Data, DataLoader as GeoLoader
from torch_geometric.nn import GCNConv, global_mean_pool


from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import xgboost as xgb
from xgboost import XGBRegressor
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# fix all random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUT = r"C:\Users\Pratham Shah\Downloads"
os.makedirs(OUT, exist_ok=True)

In [26]:
path=r"C:\Users\Pratham Shah\Downloads"
os.chdir(path)

df = pd.read_csv("microbial_multi_env_data.csv")

# Keep identifiers for community construction later
metadata = df[["species_id", "broth_id"]].copy()


X_full = df.drop(["suppression_score", "species_id", "broth_id"], axis=1).copy()
X_full = pd.get_dummies(X_full, columns=["carbon_source"], dtype=int)
y_full = df["suppression_score"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=SEED
)

# best params from grid search 3
best_params = {
    "max_depth": 2,
    "learning_rate": 0.25,
    "gamma": 0,
    "reg_lambda": 5,
}

reg_xgb_final = XGBRegressor(
    objective="reg:squarederror",
    seed=SEED,
    early_stopping_rounds=10,
    eval_metric="rmse",
    n_estimators=500,        # early stopping will pick the right number
    **best_params,
)

reg_xgb_final.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=True,
)

y_pred_test = reg_xgb_final.predict(X_test)
print(f"  XGBoost Test R²  : {r2_score(y_test, y_pred_test):.4f}")
print(f"  XGBoost Test MAE : {mean_absolute_error(y_test, y_pred_test):.4f}")
print(f"  Best iteration   : {reg_xgb_final.best_iteration}")

# predict scores for EVERY row in the full dataset
# This gives us a score for every (species, broth) combination.
df["xgb_score"] = reg_xgb_final.predict(X_full)

print("\n  Sample of scores added to df:")
print(df[["species_id", "broth_id", "xgb_score", "suppression_score"]].head(8).to_string(index=False))

[0]	validation_0-rmse:0.15602
[1]	validation_0-rmse:0.13623
[2]	validation_0-rmse:0.12317
[3]	validation_0-rmse:0.11288
[4]	validation_0-rmse:0.10426
[5]	validation_0-rmse:0.09751
[6]	validation_0-rmse:0.08993
[7]	validation_0-rmse:0.08447
[8]	validation_0-rmse:0.07960
[9]	validation_0-rmse:0.07616
[10]	validation_0-rmse:0.07275
[11]	validation_0-rmse:0.06967
[12]	validation_0-rmse:0.06738
[13]	validation_0-rmse:0.06561
[14]	validation_0-rmse:0.06425
[15]	validation_0-rmse:0.06380
[16]	validation_0-rmse:0.06273
[17]	validation_0-rmse:0.06212
[18]	validation_0-rmse:0.06116
[19]	validation_0-rmse:0.06085
[20]	validation_0-rmse:0.06041
[21]	validation_0-rmse:0.06071
[22]	validation_0-rmse:0.06034
[23]	validation_0-rmse:0.06023
[24]	validation_0-rmse:0.05978
[25]	validation_0-rmse:0.05867
[26]	validation_0-rmse:0.05887
[27]	validation_0-rmse:0.05811
[28]	validation_0-rmse:0.05805
[29]	validation_0-rmse:0.05771
[30]	validation_0-rmse:0.05764
[31]	validation_0-rmse:0.05784
[32]	validation_0-

In [27]:
print("Defining communities (one per broth)")


broths = sorted(df["broth_id"].unique())      # ['Broth_1', ..., 'Broth_10']
species_list = sorted(df["species_id"].unique())  # ['Species_1', ..., 'Species_50']

print(f"  Broths (= communities) : {len(broths)}")
print(f"  Species per community  : {len(species_list)}")
print(f"  Total graphs to build  : {len(broths)}")

Defining communities (one per broth)
  Broths (= communities) : 10
  Species per community  : 50
  Total graphs to build  : 10


In [28]:
print("Building the species interaction graph")


N_SPECIES = len(species_list)   # 50
edges_src=[]
edges_dst=[]

# Fully connected edge index

for i in range(N_SPECIES):
    for j in range(N_SPECIES):
        if i!=j: # no self loops
            edges_src.append(i)
            edges_dst.append(j)
      

# Convert to PyG format: shape [2, num_edges]
edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)

n_edges_undirected = len(edges_src) // 2
print(f"  Nodes (species)         : {N_SPECIES}")
print(f"  Edges (unique pairs)    : {n_edges_undirected}")
print(f"  Graph density           : {n_edges_undirected / (N_SPECIES*(N_SPECIES-1)/2):.2%}")
print(f"  edge_index shape        : {edge_index.shape}  (2 × {edge_index.shape[1]})")
print(f"  edge_index[0][:8]       : {edge_index[0][:8].tolist()}  (source nodes)")
print(f"  edge_index[1][:8]       : {edge_index[1][:8].tolist()}  (dest nodes)")

Building the species interaction graph
  Nodes (species)         : 50
  Edges (unique pairs)    : 1225
  Graph density           : 100.00%
  edge_index shape        : torch.Size([2, 2450])  (2 × 2450)
  edge_index[0][:8]       : [0, 0, 0, 0, 0, 0, 0, 0]  (source nodes)
  edge_index[1][:8]       : [1, 2, 3, 4, 5, 6, 7, 8]  (dest nodes)


In [38]:
print("Converting communities to PyG Data objects")


# species to integer index mapping (for node numbering)
species_to_idx = {sp: i for i, sp in enumerate(species_list)}

# Build node feature matrix for each broth
graph_list = []

for broth in broths:
    
    # get the subset of df for this broth 
    broth_df = df[df["broth_id"] == broth].copy()
    # Sort by species so node order is consistent across broths
    broth_df = broth_df.set_index("species_id").loc[species_list].reset_index()


    # Feature 1: XGBoost predicted suppression score
    scores = broth_df["xgb_score"].values.reshape(-1, 1)           # shape [50, 1]
    
    # Features 2-4: nutrient features
    # These are the same value for every species in this broth (same environment),
    # but each node needs its own row — so we just repeat the value 50 times.
    iron = broth_df["iron_concentration"].values.reshape(-1, 1)    # shape [50, 1]
    
    carbon_oh = pd.get_dummies(
        broth_df["carbon_source"], dtype=float
    ).reindex(columns=["acetate", "fructose", "glucose"], fill_value=0).values
    carbon_oh = carbon_oh[:, :2]   # keep acetate and fructose; glucose is the baseline
                                    # shape [50, 2]

    node_feats=np.hstack([scores,iron,carbon_oh])
    x=torch.tensor(node_feats,dtype=torch.float)
    
    # Community label — more than just the mean.
    # Logic: start from the mean, then add a small bonus for communities
    # where the TOP suppressors are very strong (max - mean captures how
    # "peaked" the distribution is — a community with one dominant suppressor
    # behaves differently from one where everyone is mediocre).
    # We also add a tiny penalty for very high variance (unstable communities).
    mean_score = float(np.mean(scores))
    top_bonus  = 0.10 * float(np.max(scores) - np.mean(scores))   # reward for elite suppressors
    var_penalty = 0.05 * float(np.std(scores))                    # mild penalty for instability
    community_score = float(np.clip(mean_score + top_bonus - var_penalty, 0, 1))
    
    y=torch.tensor([community_score], dtype=torch.float)
    
    
    # create the Data object
    data = Data(x=x, edge_index=edge_index, y=y)
    graph_list.append(data)



Converting communities to PyG Data objects


In [39]:
print(f"Total graphs in graph_list: {len(graph_list)}")

Total graphs in graph_list: 10


In [41]:
print("Train / test split and DataLoaders")

# Shuffle and split
random.shuffle(graph_list)
split_idx = int(0.8 * len(graph_list))     # 8 train, 2 test
train_graphs = graph_list[:split_idx]
test_graphs  = graph_list[split_idx:]

train_loader = GeoLoader(train_graphs, batch_size=4, shuffle=True)
test_loader  = GeoLoader(test_graphs,  batch_size=4, shuffle=False)

print(f"  Train graphs : {len(train_graphs)}")
print(f"  Test graphs  : {len(test_graphs)}")
print(f"  Train batches: {len(train_loader)}")



Train / test split and DataLoaders
  Train graphs : 8
  Test graphs  : 2
  Train batches: 2


In [49]:
print("GNN Architecture")


IN_CHANNELS  = 4    
HIDDEN_1     = 32   # neurons after GCNConv layer 1
HIDDEN_2     = 16   # neurons after GCNConv layer 2
DROPOUT_RATE = 0.2  # fraction of neurons randomly dropped during training

GNN Architecture


In [50]:
class MicrobialGNN(nn.Module):
    """
    GNN for predicting community-level pathogen suppression.
    
    Uses GATConv (Graph Attention Network) instead of GCNConv.
    
    The difference:
      GCNConv: h_i = ReLU( W · mean(neighbours) )
               Every neighbour contributes equally — fixed uniform weights.
    
      GATConv: h_i = ReLU( W · Σ α_ij · h_j )
               Each neighbour gets a learned attention weight α_ij.
               α_ij is computed from the features of both endpoints:
                   e_ij = LeakyReLU( a^T [W h_i || W h_j] )
                   α_ij = softmax over j of e_ij
               This means the model learns WHICH neighbours matter most
               for each species — biologically, which interactions are
               most important for community suppression.
    
    Parameters
    ----------
    in_channels  : number of input node features (4 here)
    hidden_1     : output size of first GAT layer
    hidden_2     : output size of second GAT layer
    heads        : number of attention heads (GAT uses multi-head attention,
                   like a transformer — each head learns a different pattern)
    dropout      : dropout probability
    """

    def __init__(self, in_channels, hidden_1, hidden_2, heads=4, dropout=0.2):
        super(MicrobialGNN, self).__init__()

        # ── GAT layer 1 ──
        # out_channels=hidden_1 PER HEAD, concat=True means heads are concatenated
        # so actual output size = hidden_1 * heads
        # e.g. hidden_1=8, heads=4 → output is 32-dimensional
        self.conv1 = GATConv(
            in_channels,
            hidden_1 // heads,   # per-head size; total output = hidden_1
            heads=heads,
            concat=True,
            dropout=dropout      # GAT applies dropout to attention coefficients
        )

        # ── GAT layer 2 ──
        # concat=False → heads are averaged, not concatenated
        # so output size = hidden_2 (not hidden_2 * heads)
        self.conv2 = GATConv(
            hidden_1,            # input = output of layer 1 = hidden_1 * heads / heads... = hidden_1
            hidden_2,
            heads=1,             # single head at the final conv layer is standard practice
            concat=False,
            dropout=dropout
        )

        self.dropout = nn.Dropout(p=dropout)
        self.lin1    = nn.Linear(hidden_2, 8)
        self.lin2    = nn.Linear(8, 1)

    def forward(self, data):
        """
        Forward pass.

        data.x          : [N_nodes_total, in_channels]
        data.edge_index : [2, N_edges_total]
        data.batch      : [N_nodes_total]  — which graph each node belongs to
        """
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # ── Stage 1: GAT layer 1 ──
        # GATConv returns (embeddings, attention_weights) when return_attention_weights=True
        # We only need embeddings here, so we call it normally
        x = F.elu(self.conv1(x, edge_index))   # ELU instead of ReLU is standard for GAT
        # shape: [N_nodes, hidden_1]            # ELU: x if x>0, α(e^x - 1) if x<0
                                                # smoother than ReLU, helps GAT training

        # ── Stage 2: GAT layer 2 ──
        x = F.elu(self.conv2(x, edge_index))   # [N_nodes, hidden_2]

        # ── Stage 3: global mean pooling ──
        x = global_mean_pool(x, batch)          # [N_graphs_in_batch, hidden_2]

        # ── Stage 4: MLP head ──
        x = self.dropout(x)
        x = F.relu(self.lin1(x))               # [N_graphs, 8]
        x = torch.sigmoid(self.lin2(x))        # [N_graphs, 1]

        return x.squeeze(-1)                   # [N_graphs]

In [51]:
model = MicrobialGNN(
    in_channels=IN_CHANNELS,
    hidden_1=HIDDEN_1,
    hidden_2=HIDDEN_2,
    dropout=DROPOUT_RATE,
)

print(f"\n  Model architecture:")
print(f"  {model}\n")

# Count total learnable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total learnable parameters: {total_params}")



  Model architecture:
  MicrobialGNN(
  (conv1): GATConv(4, 8, heads=4)
  (conv2): GATConv(32, 16, heads=1)
  (dropout): Dropout(p=0.2, inplace=False)
  (lin1): Linear(in_features=16, out_features=8, bias=True)
  (lin2): Linear(in_features=8, out_features=1, bias=True)
)

  Total learnable parameters: 929


In [52]:
print("Training the GNN")


LEARNING_RATE = 1e-3    
N_EPOCHS      = 300    

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn   = nn.MSELoss()

# Learning rate scheduler 
# After 150 epochs without improvement, reduce LR by factor 0.5
# This helps fine-tune once the model is in a good region
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=30, factor=0.5
)

train_losses = []
test_losses  = []

for epoch in range(1, N_EPOCHS + 1):

    # TRAINING PHASE
    model.train()           # enable dropout
    total_train_loss = 0.0
    n_train_batches  = 0

    for batch in train_loader:
        optimizer.zero_grad()               # 1. reset gradients

        pred = model(batch)                 # 2. forward pass
        loss = loss_fn(pred, batch.y.squeeze())  # 3. compute loss

        loss.backward()                     # 4. backward pass
        optimizer.step()                    # 5. update weights

        total_train_loss += loss.item()
        n_train_batches  += 1

    avg_train_loss = total_train_loss / n_train_batches

    # EVALUATION PHASE 
    model.eval()            # disable dropout
    total_test_loss = 0.0
    n_test_batches  = 0

    with torch.no_grad():   
        for batch in test_loader:
            pred = model(batch)
            loss = loss_fn(pred, batch.y.squeeze())
            total_test_loss += loss.item()
            n_test_batches  += 1

    avg_test_loss = total_test_loss / n_test_batches

    train_losses.append(avg_train_loss)
    test_losses.append(avg_test_loss)

    # Update learning rate based on test loss
    scheduler.step(avg_test_loss)

    # Print progress every 50 epochs
    if epoch % 50 == 0 or epoch == 1:
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch:>3}/{N_EPOCHS}  |  "
              f"Train MSE: {avg_train_loss:.5f}  |  "
              f"Test MSE: {avg_test_loss:.5f}  |  "
              f"LR: {current_lr:.6f}")

Training the GNN
  Epoch   1/300  |  Train MSE: 0.04721  |  Test MSE: 0.02144  |  LR: 0.001000
  Epoch  50/300  |  Train MSE: 0.01179  |  Test MSE: 0.00755  |  LR: 0.001000
  Epoch 100/300  |  Train MSE: 0.00199  |  Test MSE: 0.00270  |  LR: 0.001000
  Epoch 150/300  |  Train MSE: 0.00074  |  Test MSE: 0.00080  |  LR: 0.000500
  Epoch 200/300  |  Train MSE: 0.00363  |  Test MSE: 0.00135  |  LR: 0.000125
  Epoch 250/300  |  Train MSE: 0.00157  |  Test MSE: 0.00106  |  LR: 0.000063
  Epoch 300/300  |  Train MSE: 0.00105  |  Test MSE: 0.00091  |  LR: 0.000016


In [53]:


print("Evaluation & Results")


model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        pred = model(batch)
        p = pred.numpy()
        t = batch.y.squeeze().numpy()
        all_preds.extend(p if p.ndim > 0 else [float(p)])
        all_true.extend(t if t.ndim > 0 else [float(t)])

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

r2  = r2_score(all_true, all_preds) if len(all_true) > 1 else float("nan")
mae = mean_absolute_error(all_true, all_preds)
mse = np.mean((all_preds - all_true) ** 2)

print(f"\n  Test MSE : {mse:.5f}")
print(f"  Test MAE : {mae:.5f}")
print(f"  Test R²  : {r2:.4f}  (note: only {len(all_true)} test graphs — interpret with caution)")
print(f"\n  Predictions vs ground truth:")
print(f"  {'Community':<14} {'Predicted':>10} {'True':>10} {'Error':>10}")
for i, (p, t) in enumerate(zip(all_preds, all_true)):
    print(f"  {'Community '+str(i+1):<14} {p:>10.4f} {t:>10.4f} {abs(p-t):>10.4f}")

Evaluation & Results

  Test MSE : 0.00091
  Test MAE : 0.02512
  Test R²  : 0.8907  (note: only 2 test graphs — interpret with caution)

  Predictions vs ground truth:
  Community       Predicted       True      Error
  Community 1        0.5188     0.5607     0.0418
  Community 2        0.3698     0.3782     0.0084


In [55]:

print("Generating visualisations")

TEAL   = "#1a6b6b"
ORANGE = "#d4720c"
NAVY   = "#1c2e4a"
LIGHT  = "#e8f4f4"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "sans-serif", "font.size": 11,
})

# ─────────────────────────────────────────
# Figure 1 — XGBoost results
# ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Stage 1 — XGBoost: Single-Species Suppression", fontsize=14, fontweight="bold")

# Pred vs actual
axes[0].scatter(y_test, y_pred_test, alpha=0.7, color=TEAL, edgecolors="white", s=55)
lo, hi = min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())
axes[0].plot([lo, hi], [lo, hi], "k--", lw=1.5, label="Perfect fit")
axes[0].set_title(f"Predicted vs Actual\nTest R²={r2_score(y_test, y_pred_test):.3f}  MAE={mean_absolute_error(y_test, y_pred_test):.3f}", fontweight="bold")
axes[0].set_xlabel("Actual suppression score")
axes[0].set_ylabel("Predicted suppression score")
axes[0].legend()

# Feature importance
importances = reg_xgb_final.feature_importances_
feat_df = pd.DataFrame({"Feature": X_full.columns, "Importance": importances})
feat_df = feat_df.sort_values("Importance", ascending=False).head(10)
nutrient_feats = {"iron_concentration", "carbon_source_acetate", "carbon_source_fructose", "carbon_source_glucose"}
bar_colors = [ORANGE if f in nutrient_feats else TEAL for f in feat_df["Feature"]]
axes[1].barh(feat_df["Feature"].iloc[::-1], feat_df["Importance"].iloc[::-1],
             color=bar_colors[::-1], edgecolor="white")
axes[1].set_title("Top Feature Importances\n(orange = nutrient, teal = genomic)", fontweight="bold")
axes[1].set_xlabel("Importance score")

plt.tight_layout()
plt.savefig(f"{OUT}/fig1.2_xgboost_results.png", dpi=150, bbox_inches="tight")

plt.show()
print("  → fig1_xgboost_results.png")

# ─────────────────────────────────────────
# Figure 2 — XGBoost scores per species
# ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle("XGBoost Predicted Scores — Input to GNN Nodes", fontsize=14, fontweight="bold")

# Show score for each species across all broths (as box plots)
score_pivot = df.pivot(index="species_id", columns="broth_id", values="xgb_score")
mean_scores = score_pivot.mean(axis=1).sort_values()
sorted_species = mean_scores.index.tolist()

bp = ax.boxplot(
    [score_pivot.loc[sp].values for sp in sorted_species],
    vert=True, patch_artist=True,
    medianprops=dict(color=ORANGE, linewidth=2),
    boxprops=dict(facecolor=LIGHT, color=TEAL),
    whiskerprops=dict(color=TEAL), capprops=dict(color=TEAL),
    flierprops=dict(marker="o", color=TEAL, alpha=0.4, markersize=3),
)
ax.set_xticks(range(1, len(sorted_species) + 1))
ax.set_xticklabels([s.replace("Species_","S") for s in sorted_species], rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Species (sorted by mean predicted score)")
ax.set_ylabel("XGBoost predicted suppression score")
ax.set_title("Each box = score variation across 10 broth conditions\nThis is what the GNN sees as node features", fontweight="bold")
ax.axhline(0.5, ls="--", color="grey", lw=1, alpha=0.5, label="0.5 threshold")
ax.legend()

plt.tight_layout()
plt.savefig(f"{OUT}/fig2.2_xgb_scores_per_species.png", dpi=150, bbox_inches="tight")

plt.show()
print("  → fig2_xgb_scores_per_species.png")

# ─────────────────────────────────────────
# Figure 3 — GNN training curves
# ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Stage 2 — GNN Training", fontsize=14, fontweight="bold")

epochs_range = range(1, len(train_losses) + 1)
axes[0].plot(epochs_range, train_losses, color=TEAL,   lw=2, label="Train MSE")
axes[0].plot(epochs_range, test_losses,  color=ORANGE, lw=2, label="Test MSE")
axes[0].set_title("Loss Curves (MSE)", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zoomed last 100 epochs
axes[1].plot(epochs_range[-150:], train_losses[-150:], color=TEAL,   lw=2, label="Train MSE")
axes[1].plot(epochs_range[-150:], test_losses[-150:],  color=ORANGE, lw=2, label="Test MSE")
axes[1].set_title("Loss Curves — Final 150 Epochs (zoomed)", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/fig3.2_gnn_training_curves.png", dpi=150, bbox_inches="tight")

plt.show()
print("  → fig3_gnn_training_curves.png")

# ─────────────────────────────────────────
# Figure 4 — Node embedding visualisation
# (what the GNN learned about each species)
# ─────────────────────────────────────────
"""
After training, we can look at what the GNN's internal representation
(node embeddings after the 2 GCN layers) looks like for each species.
We extract these for the first community (Broth_1) and project them
to 2D using PCA.  Species with similar embeddings cluster together,
revealing which species the GNN groups as "functionally similar".
"""
from sklearn.decomposition import PCA

model.eval()
with torch.no_grad():
    # Manually run just the GCN layers (not pooling or MLP) for graph 0
    d = graph_list[0]
    x_emb = F.elu(model.conv1(d.x, d.edge_index))
    x_emb = F.elu(model.conv2(x_emb, d.edge_index))
    x_emb = x_emb.numpy()   # [50, 16]

pca = PCA(n_components=2)
coords = pca.fit_transform(x_emb)   # [50, 2]

# Colour by the XGBoost score of each species under Broth_1
broth1_scores = graph_list[0].x[:, 0].numpy()

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(coords[:, 0], coords[:, 1], c=broth1_scores,
                cmap="RdYlGn", s=80, edgecolors="white", linewidths=0.5)
plt.colorbar(sc, ax=ax, label="XGBoost suppression score")
ax.set_title("GNN Node Embeddings (PCA projection, Broth_1)\n"
             "Colour = XGBoost score. Clusters = species the GNN sees as functionally similar",
             fontweight="bold")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")

# Label a few extremes
top_idx = np.argsort(broth1_scores)[-3:]
bot_idx = np.argsort(broth1_scores)[:3]
for idx in np.concatenate([top_idx, bot_idx]):
    ax.annotate(species_list[idx].replace("Species_", "S"),
                (coords[idx, 0], coords[idx, 1]),
                fontsize=8, ha="left", va="bottom",
                xytext=(3, 3), textcoords="offset points")

plt.tight_layout()
plt.savefig(f"{OUT}/fig4.2_node_embeddings_pca.png", dpi=150, bbox_inches="tight")

plt.show()
print("  → fig4_node_embeddings_pca.png")

# ─────────────────────────────────────────
# Figure 5 — Community scores heatmap
# (what does each community predict vs its constituent species scores)
# ─────────────────────────────────────────
model.eval()
community_results = []
with torch.no_grad():
    for i, (broth, data) in enumerate(zip(broths, graph_list)):
        d = Data(x=data.x, edge_index=data.edge_index,
                 batch=torch.zeros(data.x.shape[0], dtype=torch.long))
        pred_score  = model(d).item()
        mean_single = data.x[:, 0].mean().item()   # mean XGBoost score
        true_score  = data.y.item()
        community_results.append({
            "broth": broth,
            "mean_single_score": mean_single,
            "predicted_community": pred_score,
            "true_community": true_score,
        })

res_df = pd.DataFrame(community_results)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Community-Level Predictions vs Single-Species Scores", fontsize=14, fontweight="bold")

x_pos = np.arange(len(res_df))
width = 0.28

axes[0].bar(x_pos - width, res_df["mean_single_score"], width, label="Mean single-species (XGBoost)", color=TEAL,   edgecolor="white")
axes[0].bar(x_pos,          res_df["true_community"],   width, label="True community score",           color=LIGHT,  edgecolor=NAVY, linewidth=1.2)
axes[0].bar(x_pos + width, res_df["predicted_community"], width, label="GNN predicted",               color=ORANGE, edgecolor="white")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([b.replace("Broth_","B") for b in res_df["broth"]], rotation=45)
axes[0].set_ylabel("Suppression score")
axes[0].set_title("Per-Community Score Comparison", fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1)

# Scatter: mean single-species vs predicted community
axes[1].scatter(res_df["mean_single_score"], res_df["predicted_community"],
                s=80, color=TEAL, edgecolors="white", zorder=3, label="Communities")
lo2 = min(res_df["mean_single_score"].min(), res_df["predicted_community"].min()) - 0.02
hi2 = max(res_df["mean_single_score"].max(), res_df["predicted_community"].max()) + 0.02
axes[1].plot([lo2, hi2], [lo2, hi2], "k--", lw=1.2, label="y = x (no community effect)")
for _, row in res_df.iterrows():
    axes[1].annotate(row["broth"].replace("Broth_","B"),
                     (row["mean_single_score"], row["predicted_community"]),
                     fontsize=8, xytext=(3,3), textcoords="offset points")
axes[1].set_xlabel("Mean single-species XGBoost score")
axes[1].set_ylabel("GNN predicted community score")
axes[1].set_title("Community Effect: Points above y=x\nmean GNN predicts community > sum of parts", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{OUT}/fig5.2_community_predictions.png", dpi=150, bbox_inches="tight")

plt.show()
print("  → fig5_community_predictions.png")

# ─────────────────────────────────────────
# Figure 6 — Graph topology visualisation
# ─────────────────────────────────────────
import networkx as nx

fig, ax = plt.subplots(figsize=(10, 8))

G = nx.Graph()
G.add_nodes_from(range(N_SPECIES))
for s, d_node in zip(edges_src[::2], edges_dst[::2]):
    G.add_edge(s, d_node)

# Colour nodes by their mean XGBoost score across all broths
mean_xgb = df.groupby("species_id")["xgb_score"].mean()
node_colors = [mean_xgb.get(species_list[i], 0.5) for i in range(N_SPECIES)]

pos = nx.spring_layout(G, seed=SEED, k=0.4)
nodes = nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                                cmap=plt.cm.RdYlGn, node_size=120,
                                ax=ax, vmin=0, vmax=1)
nx.draw_networkx_edges(G, pos, alpha=0.2, width=0.5, ax=ax)
nx.draw_networkx_labels(G, pos,
                         labels={i: species_list[i].replace("Species_","") for i in range(N_SPECIES)},
                         font_size=6, ax=ax)
plt.colorbar(nodes, ax=ax, label="Mean XGBoost suppression score")
ax.set_title(f"Community Interaction Graph — {N_SPECIES} species, {n_edges_undirected} edges\n"
             f"Colour = mean suppression ability. Layout = spring (approximate centrality)",
             fontweight="bold")
ax.axis("off")

plt.tight_layout()
plt.savefig(f"{OUT}/fig6.2_interaction_graph.png", dpi=150, bbox_inches="tight")

plt.show()
print("  → fig6_interaction_graph.png")

# ─────────────────────────────────────────
# Summary printout (CLEANED VERSION)
# ─────────────────────────────────────────
print("\n" + "=" * 65)
print("  PIPELINE COMPLETE — SUMMARY")
print("=" * 65)

print(f"\n  XGBOOST (single-species)")
print(f"    R²  = {r2_score(y_test, y_pred_test):.4f}")
print(f"    MAE = {mean_absolute_error(y_test, y_pred_test):.4f}")

print(f"\n  GNN (community-level, {len(test_graphs)} test graphs)")
print(f"    Final train MSE = {train_losses[-1]:.5f}")
print(f"    Final test  MSE = {test_losses[-1]:.5f}")
print(f"    Test  MAE       = {mae:.5f}")

print(f"\n  Community predictions:")
print(res_df[["broth", "mean_single_score", "predicted_community", "true_community"]].to_string(index=False))

# Figure 4b — GAT Attention Weight Distribution
model.eval()
with torch.no_grad():
    d = graph_list[0]
    # return_attention_weights=True gives (embeddings, (edge_index, alpha))
    _, (att_edge_index, att_weights_l1) = model.conv1(
        d.x, d.edge_index, return_attention_weights=True
    )
    # att_weights_l1 shape: [num_edges, num_heads]
    # average across 4 heads to get one weight per edge
    att_avg = att_weights_l1.mean(dim=1).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("GAT Layer 1 — Learned Attention Weights (Broth_1)", fontweight="bold")

# Left: histogram of all attention weights
axes[0].hist(att_avg, bins=40, color=TEAL, edgecolor="white")
axes[0].axvline(1/50, color=ORANGE, lw=2, ls="--",
                label=f"GCN baseline (uniform = {1/50:.3f})")
axes[0].set_xlabel("Attention weight α_ij")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Attention Weights\n"
                  "If GAT = GCN, all weights = 1/50", fontweight="bold")
axes[0].legend()

# Right: top-10 most-attended-to species (highest incoming attention sum)
import pandas as pd
att_df = pd.DataFrame({
    "dst": att_edge_index[1].numpy(),
    "weight": att_avg
})
top_species = att_df.groupby("dst")["weight"].sum().sort_values(ascending=False).head(10)
sp_labels = [species_list[i].replace("Species_", "S") for i in top_species.index]
axes[1].barh(sp_labels[::-1], top_species.values[::-1], color=ORANGE, edgecolor="white")
axes[1].set_title("Top 10 Most-Attended Species\n(highest total incoming attention)", fontweight="bold")
axes[1].set_xlabel("Sum of incoming attention weights")

plt.tight_layout()
plt.savefig(f"{OUT}/fig4b_gat_attention_weights.png", dpi=150, bbox_inches="tight")
plt.show()


Generating visualisations
  → fig1_xgboost_results.png
  → fig2_xgb_scores_per_species.png
  → fig3_gnn_training_curves.png
  → fig4_node_embeddings_pca.png
  → fig5_community_predictions.png
  → fig6_interaction_graph.png

  PIPELINE COMPLETE — SUMMARY

  XGBOOST (single-species)
    R²  = 0.8910
    MAE = 0.0454

  GNN (community-level, 2 test graphs)
    Final train MSE = 0.00105
    Final test  MSE = 0.00091
    Test  MAE       = 0.02512

  Community predictions:
   broth  mean_single_score  predicted_community  true_community
 Broth_1           0.577419             0.586440        0.591584
Broth_10           0.440861             0.421839        0.453260
 Broth_2           0.398975             0.418212        0.408922
 Broth_3           0.220700             0.252683        0.227069
 Broth_4           0.561616             0.591408        0.575029
 Broth_5           0.253504             0.281864        0.260547
 Broth_6           0.367315             0.383276        0.375995
 Broth_7